# Embryo Stage Classification — CNN + LSTM 


In [27]:

import os
import multiprocessing
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


num_cpus = multiprocessing.cpu_count()
print("CPUs:", num_cpus)

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
CPUs: 4


In [28]:

image_root      = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset"
annotation_root = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations"

In [29]:

class_map = {
    'tPB2': 0, 'tPNa': 1, 'tPNf': 2,  't2':  3,  't3':  4,
    't4':   5, 't5':   6, 't6':   7,  't7':  8,  't8':  9,
    't9+': 10, 'tM':  11, 'tSB': 12,  'tB': 13,
    'tEB': 14, 'tHB': 14          # tHB merged into tEB
}
num_classes = 15

## Build Sequences 

This function builds a **per-frame label map** first and then slides a window over the **entire
video timeline**. Every consecutive frame is preserved; no frames between phases are skipped.

Subsetting is done at the **video level** — a fraction of videos is selected, but every
annotated frame within each selected video is kept.

In [30]:


def build_sequences(image_root, annotation_root, seq_len=6, max_videos_ratio=0.75):
    sequences = []

    files     = [f for f in os.listdir(annotation_root) if f.endswith(".csv")]
    num_videos = int(len(files) * max_videos_ratio)
    files      = files[:num_videos]

    print(f"Using {len(files)} videos")

    for file in tqdm(files, desc="Building sequences"):
        embryo_id  = file.replace("_phases.csv", "")
        csv_path   = os.path.join(annotation_root, file)
        img_folder = os.path.join(image_root, embryo_id)

        if not os.path.exists(img_folder):
            continue

        images = sorted(os.listdir(img_folder))
        if len(images) < seq_len:
            continue

        df = pd.read_csv(csv_path, header=None)
        df.columns = ["phase", "start", "end"]

       
        frame_label = {}
        for _, row in df.iterrows():
            label = class_map.get(row["phase"])
            if label is None:
                continue
            for fi in range(int(row["start"]), int(row["end"]) + 1):
                frame_label[fi] = label

      
        timeline = []
        for fi, fname in enumerate(images):
            if fi in frame_label:
                timeline.append(
                    (os.path.join(img_folder, fname), frame_label[fi])
                )

        if len(timeline) < seq_len:
            continue

      
        for i in range(0, len(timeline) - seq_len + 1):
            window      = timeline[i : i + seq_len]
            seq_paths   = [w[0] for w in window]
            label       = window[-1][1]   # label of the last frame
            sequences.append((seq_paths, label, embryo_id))

    return sequences

In [31]:

def balanced_reduce(samples, max_per_class=1500):
    import random
    pool = {i: [] for i in range(num_classes)}
    for s in samples:
        pool[s[1]].append(s)

    reduced = []
    for cls in range(num_classes):
        cls_samples = pool[cls]
        if len(cls_samples) > max_per_class:
            cls_samples = random.sample(cls_samples, max_per_class)
        reduced.extend(cls_samples)

    random.shuffle(reduced)
    return reduced

In [32]:

def split_data(samples):
    embryo_ids = list(set([s[2] for s in samples]))
    train_ids, temp_ids = train_test_split(embryo_ids, test_size=0.3, random_state=42)
    val_ids,  test_ids  = train_test_split(temp_ids,   test_size=0.5, random_state=42)

    train = [s for s in samples if s[2] in set(train_ids)]
    val   = [s for s in samples if s[2] in set(val_ids)]
    test  = [s for s in samples if s[2] in set(test_ids)]

    assert len(set(train_ids) & set(val_ids))  == 0, "Train/Val leakage!"
    assert len(set(train_ids) & set(test_ids)) == 0, "Train/Test leakage!"
    assert len(set(val_ids)   & set(test_ids)) == 0, "Val/Test leakage!"
    print("No data leakage confirmed.")

    return train, val, test

In [33]:

class SequenceDataset(Dataset):
    def __init__(self, sequences, transform):
        self.sequences = sequences
        self.transform = transform

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        for _ in range(3):   # retry on corrupt file
            try:
                seq_paths, label, _ = self.sequences[idx]

                frames = []
                for path in seq_paths:
                    img = Image.open(path).convert("RGB")
                    img = self.transform(img)
                    frames.append(img)

                frames = torch.stack(frames)
                return frames, torch.tensor(label, dtype=torch.long)

            except Exception:
                idx = (idx + 1) % len(self.sequences)

       
        seq_paths, _, _ = self.sequences[0]
        return torch.zeros(len(seq_paths), 3, 224, 224), torch.tensor(0, dtype=torch.long)

In [34]:

#  TRANSFORMS

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [35]:

# CLASS WEIGHTS

def compute_class_weights(samples):
    labels  = [s[1] for s in samples]
    weights = compute_class_weight(
        'balanced', classes=np.arange(num_classes), y=np.array(labels))
    return torch.tensor(weights, dtype=torch.float32).to(device)

In [36]:

# LOSS (CE + DISTANCE + EMD)


class FinalLoss(nn.Module):
    def __init__(self, class_weights, lambda_dist=0.05, lambda_emd=0.05):
        super().__init__()
        self.ce          = nn.CrossEntropyLoss(weight=class_weights)
        self.lambda_dist = lambda_dist
        self.lambda_emd  = lambda_emd

    def forward(self, outputs, targets):
        probs = torch.softmax(outputs, dim=1)
        ce    = self.ce(outputs, targets)

        idx      = torch.arange(outputs.size(1)).float().to(outputs.device)
        expected = torch.sum(probs * idx, dim=1)
        dist     = torch.abs(expected - targets.float()).mean()

        cdf_pred = torch.cumsum(probs, dim=1)
        cdf_true = torch.cumsum(
            nn.functional.one_hot(targets, outputs.size(1)).float(), dim=1)
        emd = torch.mean((cdf_pred - cdf_true) ** 2)

        return ce + self.lambda_dist * dist + self.lambda_emd * emd

In [37]:
# MODEL — MobileNetV2 CNN + Bi-LSTM

class CNN_LSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        base     = models.mobilenet_v2(weights="IMAGENET1K_V1")
        self.cnn = base.features

        # Freeze most layers
        for p in self.cnn.parameters():
            p.requires_grad = False

        # Unfreeze last 2 blocks for fine-tuning
        for p in self.cnn[-2:].parameters():
            p.requires_grad = True

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.lstm = nn.LSTM(
            input_size=1280,
            hidden_size=512,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)
        self.fc      = nn.Linear(1024, num_classes)   # 512 * 2 (bidirectional)

    def forward(self, x):
        B, T, C, H, W = x.shape

        x    = x.reshape(B * T, C, H, W)
        feat = self.cnn(x)
        feat = self.pool(feat).view(B, T, -1)         # (B, T, 1280)

        lstm_out, _ = self.lstm(feat)                 # (B, T, 1024)

        out = self.dropout(lstm_out[:, -1, :])        # last timestep
        out = self.fc(out)
        return out

## Main Execution

`seq_len=6` is intentionally smaller than the original `seq_len=8` — the timeline-aware
approach produces many more sequences (it spans all phases, not just within one), so a shorter
window keeps total memory manageable while still giving the LSTM enough temporal context.

Videos can be reduced via `max_videos_ratio`; frames inside selected videos are **never** removed.

In [38]:

# TRAINING / EVALUATION FUNCTIONS

scaler = torch.amp.GradScaler('cuda')

def train_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0

    for i, (images, labels) in enumerate(loader):
        if i % 100 == 0:
            print(f"  Batch {i}/{len(loader)}")

        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.amp.autocast(device_type='cuda'):
            outputs = model(images)
            loss    = loss_fn(outputs, labels)

        scaler.scale(loss).backward()

        # Gradient clipping for LSTM stability
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            with torch.amp.autocast(device_type='cuda'):
                outputs = model(images)
                preds   = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    return correct / total

In [39]:

sequences = build_sequences(
    image_root,
    annotation_root,
    seq_len=6,             
                           
                           
    max_videos_ratio=0.55  
)

print("Total sequences:", len(sequences))



# SPLIT BY EMBRYO


train_seq, val_seq, test_seq = split_data(sequences)

print(f"Train: {len(train_seq)} | Val: {len(val_seq)} | Test: {len(test_seq)}")

# Downsample over-represented classes; never remove frames within a window
train_seq = balanced_reduce(train_seq, 400)
val_seq   = balanced_reduce(val_seq,   100)
test_seq  = balanced_reduce(test_seq,  100)



train_loader = DataLoader(
    SequenceDataset(train_seq, train_transform),
    batch_size=8,       
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    SequenceDataset(val_seq, test_transform),
    batch_size=8,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    SequenceDataset(test_seq, test_transform),
    batch_size=8,
    num_workers=2,
    pin_memory=True
)



# MODEL + LOSS


model = CNN_LSTM(num_classes).to(device)

class_weights = compute_class_weights(train_seq)
loss_fn       = FinalLoss(class_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)


# TRAINING LOOP


best_val_acc = 0
patience     = 3
no_improve   = 0
epochs       = 5

for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")

    loss    = train_epoch(model, train_loader, loss_fn, optimizer)
    val_acc = evaluate(model, val_loader)

    print(f"Loss={loss:.4f} | Val Acc={val_acc:.4f}")

    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "lstm_best.pt")
        no_improve = 0
        print(f"  Checkpoint saved (best val acc: {best_val_acc:.4f})")
    else:
        no_improve += 1

    if no_improve >= patience:
        print("Early stopping")
        break



#TEST

model.load_state_dict(torch.load("lstm_best.pt"))

test_acc = evaluate(model, test_loader)

print("\nFINAL TEST ACCURACY:", test_acc)

Using 387 videos


Building sequences: 100%|██████████| 387/387 [00:02<00:00, 182.96it/s]


Total sequences: 159597
No data leakage confirmed.
Train: 110569 | Val: 24160 | Test: 24868

Epoch 1/5
  Batch 0/750
  Batch 100/750
  Batch 200/750
  Batch 300/750
  Batch 400/750
  Batch 500/750
  Batch 600/750
  Batch 700/750
Loss=2.3432 | Val Acc=0.2193
  Checkpoint saved (best val acc: 0.2193)

Epoch 2/5
  Batch 0/750
  Batch 100/750
  Batch 200/750
  Batch 300/750
  Batch 400/750
  Batch 500/750
  Batch 600/750
  Batch 700/750
Loss=2.1231 | Val Acc=0.2307
  Checkpoint saved (best val acc: 0.2307)

Epoch 3/5
  Batch 0/750
  Batch 100/750
  Batch 200/750
  Batch 300/750
  Batch 400/750
  Batch 500/750
  Batch 600/750
  Batch 700/750
Loss=2.0524 | Val Acc=0.2353
  Checkpoint saved (best val acc: 0.2353)

Epoch 4/5
  Batch 0/750
  Batch 100/750
  Batch 200/750
  Batch 300/750
  Batch 400/750
  Batch 500/750
  Batch 600/750
  Batch 700/750
Loss=1.9583 | Val Acc=0.2300

Epoch 5/5
  Batch 0/750
  Batch 100/750
  Batch 200/750
  Batch 300/750
  Batch 400/750
  Batch 500/750
  Batch 600/7

## Conclusion:

CNN + LSTM framework was developed to classify embryo developmental stages from sequential image data. By utilizing a sliding window sequence-building logic that preserves every consecutive frame across the entire video timeline. The CNN component effectively extracts spatial features from individual frames, while the LSTM captures temporal dependencies across the full sequence, improving the model’s ability to recognize progression patterns.
